# 04 · Human annotation, supervised baselines, final analysis (no OpenAI cost)

Run after the coders have returned their sheets to `annotation/returned/` (coder B may complete only part of the
sheet; agreement is computed on the overlap), and after whichever paid parts (01–03b) you have completed. Everything here is free; transformer baselines (optional) run on your CPU/GPU.

## Setup

In [ ]:
import os, sys
from pathlib import Path

def find_revision_dir() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, here.parent, here / "Revision", here.parent / "Revision"]:
        if (cand / "code" / "config.py").exists():
            return cand
    raise FileNotFoundError("Open this notebook from Behavioral_Economics/Revision/notebooks/")

REVISION_DIR = find_revision_dir()
PROJECT_DIR = REVISION_DIR.parent
os.environ["REVISION_DIR"] = str(REVISION_DIR)
os.environ.setdefault("DATASET_DIR", str(PROJECT_DIR / "Dataset"))
os.environ.setdefault("MAX_WORKERS", "8")   # parallel API calls; lower to 2-4 if you hit rate limits
if str(REVISION_DIR / "code") not in sys.path:
    sys.path.insert(0, str(REVISION_DIR / "code"))
print("Revision folder:", REVISION_DIR, "| Python", sys.version.split()[0])

In [ ]:
# Installs any missing packages into THIS kernel (uv, venv or conda). Does nothing if all are present.
import importlib.util, shutil, subprocess
CORE = {"openai": "openai", "pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn",
        "statsmodels": "statsmodels", "faiss": "faiss-cpu", "pypdf": "pypdf",
        "langchain_text_splitters": "langchain-text-splitters", "openpyxl": "openpyxl",
        "vaderSentiment": "vaderSentiment", "krippendorff": "krippendorff", "lingua": "lingua-language-detector",
        "matplotlib": "matplotlib", "tqdm": "tqdm", "ipywidgets": "ipywidgets"}

def _install(req_file):
    py = sys.executable
    uv = shutil.which("uv") or shutil.which("uv.exe")
    if uv:
        cmd = [uv, "pip", "install", "--python", py, "-r", str(req_file)]
    else:
        if importlib.util.find_spec("pip") is None:
            subprocess.run([py, "-m", "ensurepip", "--upgrade"], check=True)
        cmd = [py, "-m", "pip", "install", "-r", str(req_file)]
    print("Running:", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout or "")[-1500:], (r.stderr or "")[-1500:]); r.check_returncode()

missing = [pkg for mod, pkg in CORE.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing); _install(REVISION_DIR / "requirements.txt"); importlib.invalidate_caches()
print("Missing packages:", [p for m, p in CORE.items() if importlib.util.find_spec(m) is None] or "none")

In [ ]:
import importlib, json
import pandas as pd
import config as C                     # also loads ../.env
import pipeline as PL
C.ensure_dirs()
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 200)
MOCK = os.environ.get("MOCK_LLM") == "1"   # code testing only - never for results
print("OpenAI key in use :", C.key_fingerprint(), "| loaded from:", C.ENV_LOADED or "environment (no .env found)")
print("If this is not the key you expect: fix Behavioral_Economics/.env, then Restart the kernel and re-run.")

In [ ]:
RUN_TRANSFORMER_BASELINES = False   # needs: pip install -r ../requirements-transformers.txt (torch, ~1-2 GB)

## 1 · Inter-rater agreement and gold labels (R2.3, R3.2, R4.3)

In [ ]:
import annotation
print(annotation.agreement())
f = C.RESULTS_DIR / "04_human_annotation" / "inter_rater_agreement.csv"
if f.exists(): display(pd.read_csv(f))

## 2 · Supervised and transformer baselines (R2.1, R4.5)

In [ ]:
import baselines_classic as BC
gf = C.DATA_DIR / "gold_labels.csv"
if gf.exists():
    gold = pd.read_csv(gf).merge(pd.read_csv(C.DATA_DIR / "pool.csv")[["review_uid", "reviews.text"]], on="review_uid")
    BC.tfidf_cv(gold); print("TF-IDF + logistic regression (5-fold CV) done")
else:
    print("gold_labels.csv not available yet - return the coder sheets first")
if RUN_TRANSFORMER_BASELINES and not MOCK:
    import baselines_transformer as BT
    print(BT.run())

## 3 · Final analysis and figures

In [ ]:
import analysis, figures
importlib.reload(analysis); importlib.reload(figures)
analysis.run(); figures.run()
print((C.RESULTS_DIR / "06_analysis" / "STATUS.md").read_text(encoding="utf-8"))

In [ ]:
A = C.RESULTS_DIR / "06_analysis"
def show(f, **kw):
    p = A / f
    if p.exists():
        print("\n###", f); display(pd.read_csv(p, **kw))

for f in ["A_prevalence_by_system.csv", "I_systems_on_same_reviews.csv", "I_decomposition_tests.csv",
          "H_speculative_rag_by_subset.csv", "B_accuracy_vs_human.csv", "B_mcnemar_ablation_tests.csv",
          "B_calibration.csv", "C_test_retest_stability.csv", "D_llm_vs_human_dimensions.csv",
          "D_rating_correlations.csv", "D_weight_sensitivity.csv", "E_cf_focus_tests.csv",
          "F_error_rates_by_feature.csv", "G_generalisability.csv"]:
    show(f)
show("D_rds_group_comparisons.csv", index_col=0)

## 4 · Results summary for the manuscript (results/RESULTS_SUMMARY.md)

In [ ]:
import summary
importlib.reload(summary)
path = summary.run()
from IPython.display import Markdown
display(Markdown(open(path, encoding="utf-8").read()))

In [ ]:
from IPython.display import Image, display as show_img
for p in sorted((C.RESULTS_DIR / "07_figures").glob("*.png")):
    print(p.name); show_img(Image(filename=str(p), width=720))

In [ ]:
# What has been paid for so far, and what each part still costs (reads the cache only - free)
display(PL.remaining_summary())